## Feature Engineering

In [ ]:
# Handle features with more than 50 unique categories
# We use Label Encoding for features with more than 50 categories

# Create a copy for processing
df_encoded = df_copy.copy()

# Store features that need Label Encoding
label_encoding_features = []

# Store categorical features and their encoding dictionaries
categorical_features_dict = {}

# Iterate through the features to process
for feature in df_copy[['ERC20_most_rec_token_type', 'ERC20 most sent token type']].columns:
    if df_copy[feature].nunique() >= 50:  # Use Label Encoding if more than 50 unique categories
        label_encoding_features.append(feature)
        categorical_features_dict[feature] = {}  # Dictionary maps original value to encoded integer
        i = 1  # Running index (category)
        
        # Map feature values to category integers using dictionary
        for sample in df_copy[feature]:
            if sample not in categorical_features_dict[feature].keys() and sample is not np.nan:  # Replace each value (not null)
                categorical_features_dict[feature][sample] = i
                i += 1

        # Apply Label Encoding to both the copy and original data
        df_encoded[feature].replace(categorical_features_dict[feature], inplace=True)

# Print Label Encoding features and their encoding dictionaries
print("Features requiring Label Encoding:", label_encoding_features)
print("Encoding dictionaries for categorical features:", categorical_features_dict)

from sklearn.decomposition import PCA

# Perform Principal Component Analysis
pca = PCA(n_components=0.95)  # Retain 95% of variance
X_pca = pca.fit_transform(df_encoded.drop('FLAG', axis=1))

# Create Isolation Forest model
isolation_forest_model = IsolationForest(contamination=0.05, random_state=42)

# Train the model and predict anomalies
isolation_forest_preds = isolation_forest_model.fit_predict(X_pca)

# Identify anomalies
outliers_isolation_forest = isolation_forest_preds == -1

# Remove Isolation Forest detected outliers from the original data
df_encoded_cleaned = df_encoded[~outliers_isolation_forest]

# Print statistics after removing outliers
print("Data statistics after removing outliers:")
print(df_encoded_cleaned.describe())

# Assume df_copy is the DataFrame containing the data
df_pca_cleaned_isolation = df_encoded_cleaned
# Retain the 'FLAG' column and apply PCA to remaining features
features_to_pca = df_pca_cleaned_isolation.drop('FLAG', axis=1)

# Standardize the data
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_to_pca)

# Apply PCA for dimensionality reduction
pca = PCA(n_components=0.95)  # Retain 95% of variance
features_pca = pca.fit_transform(features_scaled)

# Convert reduced data back to DataFrame and add the 'FLAG' column
df_pca = pd.DataFrame(data=features_pca, columns=['PC{}'.format(i) for i in range(1, pca.n_components_ + 1)])
df_pca['FLAG'] = df_copy['FLAG']

# Print the reduced DataFrame
print(df_pca)

# Compute correlation matrix between features
corr = df_pca.corr()

# Filter features with high correlation
corr_df = corr[(corr > 0.6)].dropna(axis=1, thresh=2)
corr_df = corr_df[corr_df != 1]

# Find features with correlation above 0.9
to_drop = corr_df[corr_df > 0.9]
high_corr_features = to_drop.values[to_drop.values > 0]

# Find features to drop
to_drop_features = to_drop.unstack().sort_values(kind="quicksort")[to_drop.unstack() > 0]

print("List of features to drop:")
print(to_drop_features)

# View correlation between features and label, sorted in descending order
correlation_with_label = df_pca.corr()['FLAG'].sort_values(ascending=False)

# Print feature-label correlations
print("Feature correlations with label:")
print(correlation_with_label)

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For better readability, split the code into two functions
def plot_correlation_heatmap(data, title):
    corr = data.corr()
    mask = np.zeros_like(corr)
    mask[np.triu_indices_from(mask)] = True
    with sns.axes_style('white'):
        fig, ax = plt.subplots(figsize=(18, 10))
        sns.heatmap(corr, mask=mask, annot=False, cmap='magma', center=0, linewidths=0.1, square=True)
        plt.title(title)
        plt.show()

# Plot correlation matrices for fraud and non-fraud samples separately
fraud_sample = df_encoded_cleaned[df_encoded_cleaned['FLAG'] == 1]
non_fraud_sample = df_encoded_cleaned[df_encoded_cleaned['FLAG'] == 0]

plot_correlation_heatmap(fraud_sample, 'Fraudulant Correlation')
plot_correlation_heatmap(non_fraud_sample, 'Non-Fraudulant Correlation')

# 1. Create a copy first (keep original logic)
dropping_corr = df_encoded.copy()

# 2. Key fix: select only numeric columns for correlation (exclude string columns)
# numeric_only=True forces only numeric columns to be processed, avoiding string conversion errors
corr_matrix = df_copy.corr(numeric_only=True)

# 3. Filter columns correlated with FLAG (check if FLAG exists first)
if 'FLAG' not in corr_matrix.columns:
    raise ValueError("'FLAG' column not found in the DataFrame. Please check the column name!")

# 4. Sort by correlation with FLAG column
corr_decision = pd.DataFrame(corr_matrix['FLAG'].sort_values(ascending=False))

# 5. Filter features with very low correlation (absolute value < 0.02)
low_corr_features = corr_decision[
    (corr_decision['FLAG'] < 0.02) & (corr_decision['FLAG'] > -0.02)
].index

# 6. Drop low-correlation features (check if features exist in dropping_corr first)
# Avoid KeyError caused by missing features
low_corr_features = [feat for feat in low_corr_features if feat in dropping_corr.columns]
dropping_encoder = dropping_corr.drop(low_corr_features, axis=1)

# 7. Print results for verification
print(f"Number of features with very low correlation with FLAG (|r| < 0.02): {len(low_corr_features)}")
print("Low-correlation features:", low_corr_features)
print("\nNumber of columns after dropping low-correlation features:", dropping_encoder.shape[1])
print("Number of columns before dropping:", dropping_corr.shape[1])